In [1]:
# Import libraries
import pandas as pd
import os

# Load all processed files
BASE_PATH = "../data/processed"

deaths_df = pd.read_csv(os.path.join(BASE_PATH, "total_accidental_deaths_2015_2022.csv"))
transport_df = pd.read_csv(os.path.join(BASE_PATH, "mode_of_transport_2015_2022.csv"))
month_df = pd.read_csv(os.path.join(BASE_PATH, "month_of_occurrence_2015_2022.csv"))
time_df = pd.read_csv(os.path.join(BASE_PATH, "time_of_occurrence_2015_2022.csv"))
road_df = pd.read_csv(os.path.join(BASE_PATH, "road_classification_2015_2022.csv"))
place_df = pd.read_csv(os.path.join(BASE_PATH, "place_of_occurrence_2015_2022.csv"))

print("All files loaded!")
print(f"Deaths: {deaths_df.shape}")
print(f"Transport: {transport_df.shape}")
print(f"Month: {month_df.shape}")
print(f"Time: {time_df.shape}")
print(f"Road: {road_df.shape}")
print(f"Place: {place_df.shape}")

All files loaded!
Deaths: (280, 4)
Transport: (280, 10)
Month: (280, 15)
Time: (280, 11)
Road: (280, 8)
Place: (280, 19)


In [2]:
# Merge all files into master dataset
master_df = deaths_df.merge(transport_df, on=["S.No.", "State/UT", "Year"], how="left")
master_df = master_df.merge(month_df, on=["S.No.", "State/UT", "Year"], how="left")
master_df = master_df.merge(time_df, on=["S.No.", "State/UT", "Year"], how="left")
master_df = master_df.merge(road_df, on=["S.No.", "State/UT", "Year"], how="left")
master_df = master_df.merge(place_df, on=["S.No.", "State/UT", "Year"], how="left")

# Add extra useful columns
# Night %
night_cols = ["0000 hrs to 0300 hrs (Night)", "0300 hrs to 0600 hrs (Night)",
              "1800 hrs to 2100 hrs (Night)", "2100 hrs to 2400 hrs (Night)"]
day_cols = ["0600 hrs to 0900 hrs (Day)", "0900 hrs to 1200 hrs (Day)",
            "1200 hrs to 1500 hrs (Day)", "1500 hrs to 1800 hrs (Day)"]

master_df["Total Night Accidents"] = master_df[night_cols].sum(axis=1)
master_df["Total Day Accidents"] = master_df[day_cols].sum(axis=1)
master_df["Night %"] = (master_df["Total Night Accidents"] /
                         (master_df["Total Night Accidents"] +
                          master_df["Total Day Accidents"]) * 100).round(1)

# Two Wheeler %
vehicle_cols = ["Truck/Lorry/Mini Truck - Died", "Bus - Died", "SUV/Car/Jeep - Died",
                "Tractor - Died", "Three Wheeler/Auto Rickshaw - Died",
                "Two Wheeler - Died", "Other Motor Vehicles - Died"]
master_df["Total Vehicle Deaths"] = master_df[vehicle_cols].sum(axis=1)
master_df["Two Wheeler %"] = (master_df["Two Wheeler - Died"] /
                               master_df["Total Vehicle Deaths"] * 100).round(1)

# Season
def get_season(row):
    month_totals = {
        "Winter": row[["December", "January", "February"]].sum(),
        "Summer": row[["March", "April", "May"]].sum(),
        "Monsoon": row[["June", "July", "August", "September"]].sum(),
        "Post Monsoon": row[["October", "November"]].sum()
    }
    return max(month_totals, key=month_totals.get)

master_df["Peak Season"] = master_df.apply(get_season, axis=1)

# Climate Zone
climate_zones = {
    "Alpine/Cold": ["Himachal Pradesh", "Uttarakhand", "Jammu and Kashmir",
                    "Arunachal Pradesh", "Sikkim"],
    "Heavy Rainfall": ["Kerala", "Assam", "Meghalaya", "Manipur", "Mizoram",
                       "Nagaland", "Tripura", "West Bengal", "Goa"],
    "Tropical Wet & Dry": ["Andhra Pradesh", "Bihar", "Chhattisgarh", "Jharkhand",
                            "Karnataka", "Madhya Pradesh", "Maharashtra", "Odisha",
                            "Tamil Nadu", "Telangana", "Uttar Pradesh"],
    "Hot & Dry": ["Rajasthan", "Gujarat", "Haryana", "Punjab", "Delhi", "Chandigarh"],
    "Coastal/Humid": ["Andaman and Nicobar Islands", "Lakshadweep", "Puducherry",
                      "Dadra and Nagar Haveli and Daman and Diu"]
}
zone_map = {state: zone for zone, states in climate_zones.items() for state in states}
master_df["Climate Zone"] = master_df["State/UT"].map(zone_map)

# Population
population_data = {
    "Andaman and Nicobar Islands": 3.97, "Andhra Pradesh": 495.63,
    "Arunachal Pradesh": 15.84, "Assam": 351.07, "Bihar": 1239.26,
    "Chandigarh": 11.58, "Chhattisgarh": 292.36,
    "Dadra and Nagar Haveli and Daman and Diu": 6.86,
    "Delhi": 190.14, "Goa": 15.86, "Gujarat": 709.04,
    "Haryana": 283.57, "Himachal Pradesh": 73.31,
    "Jammu and Kashmir": 133.06, "Jharkhand": 382.93,
    "Karnataka": 673.62, "Kerala": 354.99, "Lakshadweep": 0.73,
    "Madhya Pradesh": 853.58, "Maharashtra": 1242.21,
    "Manipur": 32.37, "Meghalaya": 32.11, "Mizoram": 12.39,
    "Nagaland": 21.89, "Odisha": 463.56, "Puducherry": 16.73,
    "Punjab": 303.57, "Rajasthan": 815.03, "Sikkim": 6.91,
    "Tamil Nadu": 772.34, "Telangana": 384.67, "Tripura": 40.99,
    "Uttar Pradesh": 2352.88, "Uttarakhand": 115.18,
    "West Bengal": 1004.07
}
master_df["Population (Lakhs)"] = master_df["State/UT"].map(population_data)
master_df["Deaths per Lakh"] = (master_df["Total Accidental Deaths"] /
                                 master_df["Population (Lakhs)"]).round(1)

# Save
master_df.to_csv("../data/final/master_dataset_2015_2022.csv", index=False)

print(f"Master dataset saved!")
print(f"Shape: {master_df.shape}")
print(f"Columns: {list(master_df.columns)}")
print(f"\nMissing values: {master_df.isnull().sum().sum()}")
print(f"\nFirst 3 rows:")
print(master_df.head(3))

Master dataset saved!
Shape: (280, 61)
Columns: ['S.No.', 'State/UT', 'Year', 'Total Accidental Deaths', 'Truck/Lorry/Mini Truck - Died', 'Bus - Died', 'SUV/Car/Jeep - Died', 'Tractor - Died', 'Three Wheeler/Auto Rickshaw - Died', 'Two Wheeler - Died', 'Other Motor Vehicles - Died', 'January', 'February', 'March', 'April', 'May', 'June', 'July', 'August', 'September', 'October', 'November', 'December', '0000 hrs to 0300 hrs (Night)', '0300 hrs to 0600 hrs (Night)', '0600 hrs to 0900 hrs (Day)', '0900 hrs to 1200 hrs (Day)', '1200 hrs to 1500 hrs (Day)', '1500 hrs to 1800 hrs (Day)', '1800 hrs to 2100 hrs (Night)', '2100 hrs to 2400 hrs (Night)', 'National Highways - Died', 'State Highways - Died', 'Expressways - Died', 'Other Roads - Died', 'Total - Died', 'Rural Area - School/College - Total', 'Rural Area - Residential - Total', 'Rural Area - Religious - Total', 'Rural Area - Recreation - Total', 'Rural Area - Factory - Total', 'Rural Area - Others - Total', 'Rural Area - Sub Total', 

In [3]:
# Check missing values
print("Missing values per column:")
missing = master_df.isnull().sum()
print(missing[missing > 0])

Missing values per column:
Two Wheeler %    4
dtype: int64


In [4]:
# Fix missing Two Wheeler %
master_df["Two Wheeler %"] = master_df["Two Wheeler %"].fillna(0)

# Verify
print(f"Missing values after fix: {master_df.isnull().sum().sum()}")

# Save again
master_df.to_csv("../data/final/master_dataset_2015_2022.csv", index=False)
print("Master dataset saved!")

Missing values after fix: 0
Master dataset saved!
